
# **기상 데이터 및 선형회귀분석을 사용한 태양광 발전량 예측 모델**
# **(Prediction Model of Solar Power Generation Using Meteorological Data and Linear Regression Analysis)**



---
---

[사용 데이터 및 변수 설명]

- training_set.csv : 아래 변수를 1시간 간격으로 총 119시간 수집한 데이터, 분석 및 모델 구축에 사용
- test_set.csv :  아래 변수를 1시간 간격으로 총 30시간 수집한 데이터,  모델 성능평가에 사용


-  'power'    : 'Power generation (kWh)', 발전량 (종속변수)
-  'temp_air' : 'Temperature (°C)' , 기온 (독립변수)
-  'temp_gnd' : 'Ground surface temperature (°C)', 지면온도 (독립변수)
-  'humi'     : 'Relative humidity (%)', 상대습도 (독립변수)
-  'insol'    : 'Insolation (MJ/m2)', 일사량 (독립변수)

[데이터 분석, 모델 구축 및 평가 방법 요약]

1. 라이브러리 불러오기


2. 입력 데이터(종속변수, 독립변수, 트레이닝 셋, 테스트 셋) 설정
  - 입력 데이터 설정 (파일 경로, 변수명)
  - 트레이닝 셋 분류 (데이터 분석, 모델 구축)
  - 테스트 셋 분류 (모델 성능평가)


3. 트레이닝 셋에 대한 기술통계량 및 그래프 출력
  - 기술통계량 출력
  - 그래프 출력


4. 트레이닝 셋에 대한 상관계수 및 산점도 출력
  - 상관계수 출력
  - 산점도 출력


5. 트레이닝 셋을 사용한 선형회귀모델 구축 및 분석결과 출력
  - 선형회귀모델 구축
  - 선형회귀분석 결과 출력


6. 테스트 셋을 사용한 구축한 모델의 성능평가 결과 출력
  - 구축한 모델의 예측값 출력
  - r2, RMSE 계산 및 출력
  - 실제값 및 예측값 비교 그래프 출력

---
---

1. 라이브러리 불러오기


In [ ]:
# 라이브러리 불러오기
import pandas as pd                             # (기본) 데이터 프레임 사용
import numpy as np                              # (기본) 배열 계산
import matplotlib.pyplot as plt                 # (기본) 그래프 출력
import scipy.stats as ss                        # (상관분석) 상관계수 계산
import statsmodels.api as sm                    # (모델 구축) 선형회귀모델 구축
from sklearn.metrics import  r2_score           # (모델 평가) r2 계산
from sklearn.metrics import mean_squared_error  # (모델 평가) MSE(또는, RMSE) 계산

2. 입력 데이터(종속변수, 독립변수, 트레이닝 셋, 테스트 셋) 설정

In [ ]:
# 입력 데이터 설정 (파일 경로, 변수명)
path_trn = 'training_set.csv'                            # 트레이닝 셋 파일 경로 입력
path_tst = 'test_set.csv'                                # 테스트 셋 파일 경로 입력
col = ['power', 'temp_air', 'temp_gnd', 'humi', 'insol'] # 전체 변수에 대한 변수명 입력
col_y = 'power'                                          # 종속변수에 대한 변수명 입력
col_x = ['temp_air', 'temp_gnd', 'humi', 'insol']        # 독립변수에 대한 변수명 입력

# 트레이닝 셋 분류 (데이터 분석, 모델 구축)
df  = pd.read_csv(path_trn)                              # CSV 파일에서 데이터 불러옴
df = df[col]                                             # 트레이닝 셋의 전체 데이터
y = df[col_y]                                            # 트레이닝 셋의 종속변수 데이터
x = df[col_x]                                            # 트레이닝 셋의 독립변수 데이터

# 테스트 셋 분류 (모델 성능평가)
df_tst  = pd.read_csv(path_tst)                          # CSV 파일에서 데이터 불러옴
df_tst = df_tst[col]                                     # 테스트 셋의 전체 데이터
y_tst = df_tst[col_y]                                    # 테스트 셋의 종속변수 데이터
x_tst = df_tst[col_x]                                    # 테스트 셋의 독립변수 데이터

3. 트레이닝 셋에 대한 기술통계량 및 그래프 출력

In [ ]:
# 기술통계량 출력
print(df.describe().round(2).T)
print()

# 그래프 출력
for i in col:
  plt_input = df[i]                           # 입력 데이터
  plt.figure(figsize=(7, 1))                  # plot 크기 설정
  plt.plot(plt_input)                         # plot 그림
  plt.xlabel('Time (h)')                      # x축 제목 설정
  plt.ylabel(i)                               # y축 제목 설정
  plt.xticks(range(0, len(plt_input), 5))     # x축 눈금 표시값 설정
  plt.xlim([0, len(plt_input)-1])             # x축 표시범위 설정
  plt.show()                                  # plot 출력

4. 트레이닝 셋에 대한 상관계수 및 산점도 출력

In [ ]:
# 상관계수 출력
print(f"표. 각 변수들과 {col_y}간의 상관계수")
print('-'*33)
print(f"{'Variable':<14}{'r':<14}{'p':<14}")
print('-'*33)
for i in col_x:
  cor =  ss.pearsonr(df[i], df[col_y])       # 피어슨 상관분석
  print(f'{i:<14}{cor[0]:<14.3f}{cor[1]:<14.3f}')
print('-'*33)
print('')

# 산점도 출력
for i in col_x:
  plt.figure(figsize=(2, 2))
  plt.scatter(df[i], df[col_y])              # 산점도
  plt.xlabel(i)
  plt.ylabel(col_y)
  plt.show()

5. 트레이닝 셋을 사용한 선형회귀모델 구축 및 분석결과 출력

In [ ]:
# 선형회귀모델 구축
x = sm.add_constant(x)         # 트레이닝 셋의 독립변수에 상수항 추가
md = sm.OLS(y, x).fit()        # 선형회귀모델 구축

# 선형회귀분석 결과 출력
print(md.summary())            # 선형회귀분석 결과 출력
print('')
var = list(md.params.index)    # 변수 추출
B = list(md.params.round(4))   # 회귀계수 추출
print(f'* 구축한 선형회귀모델')
for i in range(len(var)):
  if i == 0:
    print(f'{col_y} = {B[0]}', end = ' ')
  else:
    if B[i] > 0:
      print(f'+ {B[i]}*{var[i]}', end = ' ')
    else:
      print(f'{B[i]}*{var[i]}', end = ' ')

6. 테스트 셋을 사용한 구축한 모델의 성능평가 결과 출력

In [ ]:
# 구축한 모델의 예측값 출력
x_tst = sm.add_constant(x_tst)              # 테스트 셋의 독립변수에 상수항 추가
y_prd = list(md.predict(x_tst))             # 테스트 셋을 사용한 모델의 예측값 출력

# r2, RMSE 계산 및 출력
r2 = r2_score(y_tst, y_prd)                 # r2 계산
MSE = mean_squared_error(y_tst, y_prd)      # MSE 계산
RMSE = np.sqrt(MSE)                         # RMSE 계산

print("표. 테스트 셋을 사용한 모델 성능평가 결과")
print('-'*40)
print(f"{'r2':<12}{'RMSE(kWh)':<12}")
print('-'*40)
print(f"{r2:<12.3f}{RMSE:<12.2f}")
print('-'*40)
print('')
print('')

# 실제값 및 예측값 비교 그래프 출력
plt.figure(figsize=(8, 3))                  # plot 크기 설정
plt.plot(y_tst, label = 'real value')       # plot 그림, label로 범례 표시값 설정 (범례!)
plt.plot(y_prd, label = 'predicted value')  # plot 그림, label로 범례 표시값 설정 (범례!)
plt.legend()                                # 범례 표시 (범례!)
plt.xlabel('Time (h)')                      # x축 제목 설정
plt.ylabel('Power generation (kWh)')        # y축 제목 설정
plt.xticks(range(0, len(y_tst), 2))         # x축 눈금 표시값 설정
plt.xlim([0, len(y_tst)-1])                 # x축 표시범위 설정
plt.show()                                  # plot 출력